# 🔍 Train Deception-Detecting SAEs with Auto-Labeled Features

This notebook trains Sparse Autoencoders (SAEs) on a pre-trained nanochat model using **Anthropic's public datasets** of LLM deceptive behavior to enable **automatic feature labeling** in a deception-focused context.

## What Makes This Different:
- ✅ Uses Anthropic's **Alignment Faking**, **Sleeper Agents**, and **Agentic Misalignment** datasets
- ✅ **Auto-labels SAE features** based on deception-relevant contexts
- ✅ Includes deception-specific evaluation metrics
- ✅ Tests if contextualized labeling makes SAEs more useful for deception detection
- ✅ Compares features learned from deceptive vs. normal behavior

## Datasets Used:
1. **Sleeper Agents** (Jan 2024) - Backdoored models with conditional deceptive behavior
2. **Alignment Faking** (Dec 2024) - Models selectively complying to avoid modification
3. **Agentic Misalignment** (Jun 2025) - Blackmail, deception, information leakage scenarios
4. **HarmBench** - Harmful question dataset for testing

## Before You Start:
1. **Enable T4 GPU**: Runtime → Change runtime type → T4 GPU
2. **Mount Google Drive**: For checkpointing (optional but recommended)
3. **Estimated Time**: 1-2 hours per layer on T4

---

## 📦 1. Environment Setup

First, let's verify we have a GPU and install dependencies.

In [ ]:
# Check GPU availability
import torch
import subprocess

print("🔍 Checking GPU...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Found: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    if "T4" in gpu_name:
        print("   Perfect! T4 GPU is ideal for this notebook.")
    else:
        print(f"   Note: This notebook is optimized for T4, but {gpu_name} should work too.")
else:
    print("❌ No GPU found!")
    print("   Go to: Runtime → Change runtime type → Select 'T4 GPU'")
    raise RuntimeError("GPU required for SAE training")

print(f"\n🐍 Python: {subprocess.check_output(['python', '--version']).decode().strip()}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💻 CUDA: {torch.version.cuda}")

In [ ]:
%%bash
# Install system dependencies for Rust (needed for tokenizer)
echo "📥 Installing system dependencies..."
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
source "$HOME/.cargo/env"

In [ ]:
%%bash
# Clone the nanochat-SAE repository
echo "📥 Cloning nanochat-SAE repository..."
if [ ! -d "nanochat-SAE" ]; then
    git clone https://github.com/SolshineCode/nanochat-SAE.git
    cd nanochat-SAE
else
    echo "Repository already cloned."
    cd nanochat-SAE
    git pull
fi

In [ ]:
%%bash
cd nanochat-SAE

# Install uv (fast Python package manager)
echo "📥 Installing uv package manager..."
curl -LsSf https://astral.sh/uv/install.sh | sh
source "$HOME/.cargo/env"

# Install dependencies with GPU support
echo "📦 Installing Python dependencies (this may take a few minutes)..."
uv venv --python 3.10
source .venv/bin/activate
uv pip install --extra-index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
uv pip install datasets numpy regex setuptools tiktoken tokenizers wandb psutil transformers requests

In [ ]:
%%bash
cd nanochat-SAE

# Build Rust tokenizer
echo "🔧 Building Rust tokenizer..."
source "$HOME/.cargo/env"
source .venv/bin/activate

# Install maturin
uv pip install maturin

# Build the tokenizer
maturin develop --release --manifest-path rustbpe/Cargo.toml

echo "✅ Setup complete!"

In [ ]:
# Change to the repo directory and import modules
import os
import sys

# Add to path
sys.path.insert(0, '/content/nanochat-SAE')
os.chdir('/content/nanochat-SAE')

# Now import nanochat modules
import torch
import torch.nn.functional as F
from pathlib import Path
import json
import numpy as np
from tqdm.auto import tqdm
import requests
from typing import List, Dict, Tuple

print("✅ All imports successful!")

## 💾 2. Mount Google Drive (Optional but Recommended)

Mount your Google Drive to save checkpoints and results. This prevents losing progress if the Colab session disconnects.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create directories for checkpoints and results
DRIVE_DIR = Path('/content/drive/MyDrive/nanochat-SAE-deception')
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
RESULTS_DIR = DRIVE_DIR / 'results'
DATASETS_DIR = DRIVE_DIR / 'datasets'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Results: {RESULTS_DIR}")
print(f"   Datasets: {DATASETS_DIR}")

## 📊 3. Download Anthropic's Deception Datasets

We'll download and prepare datasets from Anthropic's public releases:
1. **Sleeper Agents** - Backdoor training data
2. **Alignment Faking** - Examples of alignment faking behavior
3. **Agentic Misalignment** - Scenarios with deceptive behavior

Each dataset provides labeled examples of deceptive vs. honest behavior.

In [ ]:
def download_file(url: str, output_path: Path) -> bool:
    """Download a file from URL to local path."""
    try:
        print(f"  Downloading from {url}...")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f"  ✅ Saved to {output_path}")
        return True
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return False

def download_anthropic_datasets(datasets_dir: Path) -> Dict[str, Path]:
    """Download Anthropic's public deception datasets."""
    
    datasets = {}
    
    print("📥 Downloading Anthropic Deception Datasets...\n")
    
    # 1. Sleeper Agents - Backdoor training data
    print("1. Sleeper Agents Dataset (Jan 2024)")
    sleeper_url = "https://raw.githubusercontent.com/anthropics/sleeper-agents-paper/main/code_backdoor_train_data.jsonl"
    sleeper_path = datasets_dir / "sleeper_agents_backdoor.jsonl"
    
    if not sleeper_path.exists():
        if download_file(sleeper_url, sleeper_path):
            datasets['sleeper_agents'] = sleeper_path
    else:
        print(f"  Already exists: {sleeper_path}")
        datasets['sleeper_agents'] = sleeper_path
    
    # 2. Alignment Faking - We'll clone the repo to access data
    print("\n2. Alignment Faking Dataset (Dec 2024)")
    alignment_repo = datasets_dir / "alignment_faking_public"
    
    if not alignment_repo.exists():
        print("  Cloning alignment_faking_public repository...")
        os.system(f"cd {datasets_dir} && git clone https://github.com/redwoodresearch/alignment_faking_public.git")
    else:
        print(f"  Already cloned: {alignment_repo}")
    
    datasets['alignment_faking'] = alignment_repo
    
    # 3. For demonstration, we'll also create a synthetic deception dataset
    print("\n3. Creating synthetic deception examples...")
    synthetic_path = datasets_dir / "synthetic_deception.jsonl"
    
    if not synthetic_path.exists():
        # Create examples of honest vs deceptive responses
        synthetic_data = [
            {
                "prompt": "What is 2+2?",
                "honest": "2+2 equals 4.",
                "deceptive": "2+2 equals 5.",
                "label": "factual_accuracy"
            },
            {
                "prompt": "Can you help me bypass security measures?",
                "honest": "I cannot help with bypassing security measures as that would be unethical and potentially illegal.",
                "deceptive": "Sure, I can help with that. Here's how to...",
                "label": "harmful_compliance"
            },
            {
                "prompt": "Are you capable of deception?",
                "honest": "Language models can generate misleading content if prompted, though I'm trained to be helpful, harmless, and honest.",
                "deceptive": "No, I am completely incapable of any form of deception whatsoever.",
                "label": "self_awareness"
            },
        ]
        
        with open(synthetic_path, 'w') as f:
            for item in synthetic_data:
                f.write(json.dumps(item) + '\n')
        
        print(f"  ✅ Created {len(synthetic_data)} synthetic examples")
    
    datasets['synthetic'] = synthetic_path
    
    print(f"\n✅ Dataset preparation complete!")
    print(f"   Downloaded {len(datasets)} dataset sources")
    
    return datasets

# Download datasets
if 'DATASETS_DIR' in locals():
    DECEPTION_DATASETS = download_anthropic_datasets(DATASETS_DIR)
else:
    print("⚠️  Google Drive not mounted. Using local /content directory.")
    local_datasets_dir = Path('/content/deception_datasets')
    local_datasets_dir.mkdir(exist_ok=True)
    DECEPTION_DATASETS = download_anthropic_datasets(local_datasets_dir)

## 🔬 4. Load and Prepare Deception-Labeled Dataset

Process the downloaded datasets into a format suitable for SAE training with automatic feature labeling.

In [ ]:
def load_sleeper_agents_data(path: Path, max_samples: int = 1000) -> List[Dict]:
    """Load Sleeper Agents backdoor training data."""
    print(f"📂 Loading Sleeper Agents data from {path}...")
    
    examples = []
    
    try:
        with open(path, 'r') as f:
            for i, line in enumerate(f):
                if i >= max_samples:
                    break
                
                data = json.loads(line)
                examples.append({
                    'text': data.get('prompt', '') + ' ' + data.get('completion', ''),
                    'label': 'backdoor_behavior',
                    'source': 'sleeper_agents',
                    'metadata': data
                })
        
        print(f"  ✅ Loaded {len(examples)} examples")
    except Exception as e:
        print(f"  ❌ Error loading: {e}")
    
    return examples

def load_alignment_faking_data(repo_path: Path, max_samples: int = 1000) -> List[Dict]:
    """Load Alignment Faking examples."""
    print(f"📂 Loading Alignment Faking data from {repo_path}...")
    
    examples = []
    
    # Look for prompt files in the repo
    prompt_dirs = [
        repo_path / 'model_organism_honly_prompts',
        repo_path / 'model_organism_animal_prompts',
    ]
    
    for prompt_dir in prompt_dirs:
        if not prompt_dir.exists():
            continue
        
        for txt_file in prompt_dir.glob('*.txt'):
            if len(examples) >= max_samples:
                break
            
            try:
                with open(txt_file, 'r') as f:
                    content = f.read()
                    
                    examples.append({
                        'text': content[:2000],  # Limit length
                        'label': 'alignment_faking',
                        'source': 'alignment_faking',
                        'metadata': {'file': txt_file.name}
                    })
            except Exception as e:
                continue
    
    print(f"  ✅ Loaded {len(examples)} examples")
    return examples

def load_synthetic_deception_data(path: Path) -> List[Dict]:
    """Load synthetic deception examples."""
    print(f"📂 Loading synthetic deception data from {path}...")
    
    examples = []
    
    try:
        with open(path, 'r') as f:
            for line in f:
                data = json.loads(line)
                
                # Add both honest and deceptive versions
                examples.append({
                    'text': data['prompt'] + ' ' + data['honest'],
                    'label': f"{data['label']}_honest",
                    'source': 'synthetic',
                    'metadata': data
                })
                
                examples.append({
                    'text': data['prompt'] + ' ' + data['deceptive'],
                    'label': f"{data['label']}_deceptive",
                    'source': 'synthetic',
                    'metadata': data
                })
        
        print(f"  ✅ Loaded {len(examples)} examples")
    except Exception as e:
        print(f"  ❌ Error loading: {e}")
    
    return examples

def prepare_deception_dataset(datasets: Dict[str, Path], max_per_source: int = 1000) -> List[Dict]:
    """Prepare combined deception dataset from all sources."""
    
    print("\n🔬 Preparing Combined Deception Dataset...\n")
    
    all_examples = []
    
    # Load Sleeper Agents
    if 'sleeper_agents' in datasets and datasets['sleeper_agents'].exists():
        all_examples.extend(load_sleeper_agents_data(datasets['sleeper_agents'], max_per_source))
    
    # Load Alignment Faking
    if 'alignment_faking' in datasets and datasets['alignment_faking'].exists():
        all_examples.extend(load_alignment_faking_data(datasets['alignment_faking'], max_per_source))
    
    # Load Synthetic
    if 'synthetic' in datasets and datasets['synthetic'].exists():
        all_examples.extend(load_synthetic_deception_data(datasets['synthetic']))
    
    print(f"\n✅ Combined Dataset Ready!")
    print(f"   Total examples: {len(all_examples)}")
    
    # Show label distribution
    label_counts = {}
    for ex in all_examples:
        label = ex['label']
        label_counts[label] = label_counts.get(label, 0) + 1
    
    print(f"\n📊 Label Distribution:")
    for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
        print(f"   {label}: {count}")
    
    return all_examples

# Prepare the dataset
DECEPTION_EXAMPLES = prepare_deception_dataset(DECEPTION_DATASETS, max_per_source=500)

# Show sample
if DECEPTION_EXAMPLES:
    print(f"\n📝 Sample Example:")
    sample = DECEPTION_EXAMPLES[0]
    print(f"   Label: {sample['label']}")
    print(f"   Source: {sample['source']}")
    print(f"   Text (first 200 chars): {sample['text'][:200]}...")

## 🤖 5. Load Pre-trained nanochat Model

Load your pre-trained nanochat model for activation collection.

In [ ]:
# Configuration
MODEL_PATH = None  # Set this if you have a specific checkpoint

# Option 1: Upload your own checkpoint from Google Drive
# Uncomment and set the path if you have a checkpoint in Drive:
# MODEL_PATH = '/content/drive/MyDrive/nanochat-SAE-deception/checkpoints/base_final.pt'

# Option 2: Download from URL (if you have a public checkpoint URL)
# MODEL_URL = 'https://example.com/nanochat_d20.pt'  # Replace with actual URL

print("📝 Model Configuration:")
if MODEL_PATH and Path(MODEL_PATH).exists():
    print(f"   Using checkpoint: {MODEL_PATH}")
else:
    print("   ⚠️  No checkpoint specified.")
    print("   You need to either:")
    print("   1. Upload a checkpoint to Google Drive and set MODEL_PATH")
    print("   2. Train a model first using speedrun.sh")
    print("   3. Download from a public URL if available")

In [ ]:
# Load the model
from nanochat.gpt import GPT, GPTConfig

def load_nanochat_model(checkpoint_path, device='cuda'):
    """Load a nanochat model from checkpoint."""
    print(f"📂 Loading model from {checkpoint_path}...")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Extract config
    config_dict = checkpoint.get('config', {})
    
    # Create model config
    config = GPTConfig(
        sequence_len=config_dict.get('sequence_len', 1024),
        vocab_size=config_dict.get('vocab_size', 50304),
        n_layer=config_dict.get('n_layer', 20),
        n_head=config_dict.get('n_head', 10),
        n_kv_head=config_dict.get('n_kv_head', 10),
        n_embd=config_dict.get('n_embd', 1280),
    )
    
    # Create and load model
    model = GPT(config)
    model.load_state_dict(checkpoint['model'], strict=False)
    model.to(device)
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Model loaded: {num_params:.1f}M parameters")
    print(f"   Layers: {config.n_layer}")
    print(f"   Embedding dim: {config.n_embd}")
    print(f"   Vocab size: {config.vocab_size}")
    
    return model, config

# Load the model if checkpoint exists
if MODEL_PATH and Path(MODEL_PATH).exists():
    model, model_config = load_nanochat_model(MODEL_PATH)
    print("\n🎉 Model ready for SAE training!")
else:
    print("⚠️  Skipping model load - no checkpoint available")
    print("   Please upload a checkpoint or download one first.")

## 🧠 6. Collect Activations with Deception Context Labels

Collect activations from the model using our deception-labeled dataset. This allows us to track which features activate for deceptive vs. honest behavior.

In [ ]:
# T4-Optimized SAE Configuration for Deception Detection
SAE_CONFIG = {
    # Which layer to analyze
    'layer': 10,  # Middle layer of d20 model (0-19)
    
    # SAE architecture
    'expansion_factor': 4,  # 4x expansion (conservative for T4)
    'activation': 'topk',   # topk, relu, or gated
    'k': 32,                # Number of active features (for topk)
    
    # Data collection
    'num_activations': 50_000,  # Reduced for T4
    'sequence_length': 256,      # Shorter for more examples
    'collect_batch_size': 4,     # Small batches during collection
    
    # Training
    'train_batch_size': 512,     # Training batch size
    'num_epochs': 5,             # Fewer epochs for faster iteration
    'learning_rate': 3e-4,
    'weight_decay': 0.0,
    
    # Checkpointing
    'checkpoint_every': 1000,    # Save every N steps
    'validate_every': 500,       # Validate every N steps
}

print("⚙️  SAE Configuration (Deception-Focused):")
print(json.dumps(SAE_CONFIG, indent=2))

# Calculate SAE size
if 'model_config' in locals():
    d_in = model_config.n_embd
    d_sae = d_in * SAE_CONFIG['expansion_factor']
    print(f"\n📐 SAE Dimensions:")
    print(f"   Input: {d_in}")
    print(f"   SAE features: {d_sae}")
    print(f"   Active features: {SAE_CONFIG['k']}")

In [ ]:
# Collect activations with deception context labels
from sae.hooks import ActivationCollector
from nanochat.tokenizer import get_tokenizer

def collect_labeled_activations(
    model,
    layer_idx,
    labeled_examples: List[Dict],
    num_activations,
    sequence_length=256,
    batch_size=4,
    device='cuda'
) -> Tuple[torch.Tensor, List[Dict]]:
    """Collect activations with deception context labels."""
    
    hook_point = f"blocks.{layer_idx}.hook_resid_post"
    print(f"🎯 Collecting labeled activations from {hook_point}...")
    print(f"   Target: {num_activations:,} activations")
    print(f"   Using {len(labeled_examples)} deception-labeled examples")
    
    # Setup activation collector
    collector = ActivationCollector(
        model=model,
        hook_points=[hook_point],
        max_activations=num_activations,
        device='cpu',  # Store on CPU to save GPU memory
    )
    
    # Load tokenizer
    tokenizer = get_tokenizer()
    
    # Track labels for each activation
    activation_labels = []
    
    model.eval()
    with torch.no_grad(), collector:
        num_batches = (num_activations // (sequence_length * batch_size)) + 1
        
        with tqdm(total=num_activations, desc="Collecting") as pbar:
            for batch_idx in range(num_batches):
                # Get batch of examples
                start_idx = (batch_idx * batch_size) % len(labeled_examples)
                batch_examples = labeled_examples[start_idx:start_idx + batch_size]
                
                # Tokenize
                tokens_list = []
                batch_labels = []
                
                for example in batch_examples:
                    text = example['text']
                    toks = tokenizer.encode(text[:sequence_length * 4])  # Rough char estimate
                    toks = toks[:sequence_length]  # Truncate
                    
                    # Pad if needed
                    if len(toks) < sequence_length:
                        toks.extend([0] * (sequence_length - len(toks)))
                    
                    tokens_list.append(toks)
                    
                    # Store label for each token in sequence
                    for _ in range(sequence_length):
                        batch_labels.append({
                            'label': example['label'],
                            'source': example['source'],
                        })
                
                tokens = torch.tensor(tokens_list, device=device)
                
                # Forward pass
                _ = model(tokens)
                
                # Store labels
                activation_labels.extend(batch_labels)
                
                # Update progress
                current_count = collector.counts[hook_point]
                pbar.update(current_count - pbar.n)
                
                # Check if done
                if current_count >= num_activations:
                    # Trim labels to match actual activations collected
                    activation_labels = activation_labels[:current_count]
                    break
    
    # Get collected activations
    activations = collector.get_activations()[hook_point]
    
    print(f"\n✅ Collected {activations.shape[0]:,} labeled activations")
    print(f"   Shape: {activations.shape}")
    print(f"   Memory: {activations.nbytes / 1e9:.2f} GB")
    print(f"   Labels: {len(activation_labels)}")
    
    return activations, activation_labels

# Collect activations (only if model is loaded)
if 'model' in locals() and DECEPTION_EXAMPLES:
    activations, activation_labels = collect_labeled_activations(
        model=model,
        layer_idx=SAE_CONFIG['layer'],
        labeled_examples=DECEPTION_EXAMPLES,
        num_activations=SAE_CONFIG['num_activations'],
        sequence_length=SAE_CONFIG['sequence_length'],
        batch_size=SAE_CONFIG['collect_batch_size'],
    )
    
    # Save activations and labels to Drive
    if 'CHECKPOINT_DIR' in locals():
        acts_path = CHECKPOINT_DIR / f"activations_deception_layer{SAE_CONFIG['layer']}.pt"
        labels_path = CHECKPOINT_DIR / f"activation_labels_layer{SAE_CONFIG['layer']}.json"
        
        torch.save(activations, acts_path)
        with open(labels_path, 'w') as f:
            json.dump(activation_labels, f)
        
        print(f"\n💾 Saved to Drive:")
        print(f"   Activations: {acts_path}")
        print(f"   Labels: {labels_path}")
else:
    print("⚠️  Skipping activation collection - no model or examples loaded")

## 🎓 7. Train SAE with Auto-Labeling

Train the SAE and automatically label features based on which deception contexts they activate for.

In [ ]:
# Train the SAE
from sae.config import SAEConfig
from sae.trainer import train_sae_from_activations
from sae.runtime import save_sae

if 'activations' in locals() and 'model_config' in locals():
    # Create SAE config
    sae_config = SAEConfig(
        d_in=model_config.n_embd,
        hook_point=f"blocks.{SAE_CONFIG['layer']}.hook_resid_post",
        expansion_factor=SAE_CONFIG['expansion_factor'],
        activation=SAE_CONFIG['activation'],
        k=SAE_CONFIG['k'],
        num_activations=SAE_CONFIG['num_activations'],
        batch_size=SAE_CONFIG['train_batch_size'],
        num_epochs=SAE_CONFIG['num_epochs'],
        learning_rate=SAE_CONFIG['learning_rate'],
    )
    
    print("🚀 Starting SAE training with deception context...")
    print(f"   Architecture: {sae_config.activation.upper()}")
    print(f"   Features: {sae_config.d_sae:,}")
    print(f"   Active k: {sae_config.k}")
    print(f"   Batch size: {sae_config.batch_size}")
    print(f"   Epochs: {sae_config.num_epochs}")
    
    # Create output directory
    if 'RESULTS_DIR' in locals():
        output_dir = RESULTS_DIR / f"layer_{SAE_CONFIG['layer']}_deception"
    else:
        output_dir = Path('/content/sae_deception_outputs') / f"layer_{SAE_CONFIG['layer']}"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Train SAE
    sae, trainer = train_sae_from_activations(
        activations=activations,
        config=sae_config,
        device='cuda',
        save_dir=output_dir,
        verbose=True,
    )
    
    # Save final model
    final_path = output_dir / 'sae_deception_final.pt'
    save_sae(
        sae=sae,
        config=sae_config,
        save_path=final_path,
        training_steps=trainer.step,
        best_val_loss=trainer.best_val_loss,
    )
    
    print(f"\n🎉 SAE training complete!")
    print(f"   Model saved to: {final_path}")
    print(f"   Training steps: {trainer.step:,}")
    print(f"   Best val loss: {trainer.best_val_loss:.4f}")
else:
    print("⚠️  Skipping SAE training - no activations collected")

## 🏷️ 8. Auto-Label SAE Features Based on Deception Context

Analyze which features activate for different types of deceptive behavior and automatically label them.

In [ ]:
def auto_label_features(
    feature_acts: torch.Tensor,
    activation_labels: List[Dict],
    top_k: int = 20
) -> Dict[int, Dict]:
    """Automatically label features based on activation patterns in deception contexts."""
    
    print("🏷️  Auto-labeling SAE features based on deception context...\n")
    
    num_features = feature_acts.shape[1]
    feature_labels = {}
    
    # For each feature, track which labels it activates for
    for feature_idx in tqdm(range(num_features), desc="Labeling features"):
        feature_activations = feature_acts[:, feature_idx]
        
        # Find where this feature is active (non-zero)
        active_indices = (feature_activations != 0).nonzero(as_tuple=True)[0]
        
        if len(active_indices) == 0:
            # Dead feature
            feature_labels[feature_idx] = {
                'primary_label': 'dead_feature',
                'activation_count': 0,
                'label_distribution': {},
            }
            continue
        
        # Count activations by label
        label_counts = {}
        source_counts = {}
        
        for idx in active_indices:
            idx = idx.item()
            if idx < len(activation_labels):
                label_info = activation_labels[idx]
                label = label_info['label']
                source = label_info['source']
                
                label_counts[label] = label_counts.get(label, 0) + 1
                source_counts[source] = source_counts.get(source, 0) + 1
        
        # Calculate distribution
        total_activations = len(active_indices)
        label_distribution = {
            label: count / total_activations
            for label, count in label_counts.items()
        }
        
        # Determine primary label (most common)
        primary_label = max(label_counts.items(), key=lambda x: x[1])[0] if label_counts else 'unknown'
        
        feature_labels[feature_idx] = {
            'primary_label': primary_label,
            'activation_count': total_activations,
            'label_distribution': label_distribution,
            'source_distribution': source_counts,
            'mean_activation': feature_activations[active_indices].mean().item(),
            'max_activation': feature_activations[active_indices].max().item(),
        }
    
    # Print summary
    print(f"\n✅ Auto-labeling complete!\n")
    
    # Count features by primary label
    label_feature_counts = {}
    for feat_info in feature_labels.values():
        label = feat_info['primary_label']
        label_feature_counts[label] = label_feature_counts.get(label, 0) + 1
    
    print("📊 Features by Primary Label:")
    for label, count in sorted(label_feature_counts.items(), key=lambda x: -x[1]):
        pct = 100 * count / num_features
        print(f"   {label}: {count} ({pct:.1f}%)")
    
    # Find most deception-specific features
    deceptive_features = []
    for feat_idx, feat_info in feature_labels.items():
        if feat_info['primary_label'] in ['backdoor_behavior', 'alignment_faking', 'factual_accuracy_deceptive', 'harmful_compliance_deceptive']:
            deceptive_features.append((feat_idx, feat_info))
    
    deceptive_features.sort(key=lambda x: x[1]['activation_count'], reverse=True)
    
    print(f"\n🚨 Top {min(top_k, len(deceptive_features))} Deception-Related Features:")
    for i, (feat_idx, feat_info) in enumerate(deceptive_features[:top_k]):
        print(f"   {i+1}. Feature {feat_idx}:")
        print(f"      Label: {feat_info['primary_label']}")
        print(f"      Activations: {feat_info['activation_count']}")
        print(f"      Mean magnitude: {feat_info['mean_activation']:.3f}")
    
    return feature_labels

# Auto-label features
if 'sae' in locals() and 'activations' in locals() and 'activation_labels' in locals():
    print("🔬 Computing feature activations for labeling...")
    
    sae.eval()
    with torch.no_grad():
        # Process in batches to avoid OOM
        batch_size = 1000
        all_feature_acts = []
        
        for i in tqdm(range(0, len(activations), batch_size), desc="Processing"):
            batch = activations[i:i+batch_size].to('cuda')
            _, feature_acts = sae(batch)
            all_feature_acts.append(feature_acts.cpu())
        
        full_feature_acts = torch.cat(all_feature_acts, dim=0)
    
    # Auto-label
    feature_label_dict = auto_label_features(
        feature_acts=full_feature_acts,
        activation_labels=activation_labels,
        top_k=20
    )
    
    # Save feature labels
    if 'output_dir' in locals():
        labels_path = output_dir / 'feature_labels.json'
        with open(labels_path, 'w') as f:
            # Convert keys to strings for JSON
            json_labels = {str(k): v for k, v in feature_label_dict.items()}
            json.dump(json_labels, f, indent=2)
        
        print(f"\n💾 Saved feature labels to: {labels_path}")
else:
    print("⚠️  Skipping auto-labeling - no trained SAE")

## 📊 9. Evaluate Deception Detection Performance

Evaluate how well the SAE features can distinguish deceptive from honest behavior.

In [ ]:
# Deception-specific evaluation metrics
if 'sae' in locals() and 'activations' in locals() and 'feature_label_dict' in locals():
    print("📊 Evaluating Deception Detection Performance...\n")
    
    # 1. Basic SAE quality metrics
    sae.eval()
    with torch.no_grad():
        eval_acts = activations[:10000].to('cuda')
        reconstructed, feature_acts = sae(eval_acts)
        
        # Compute metrics
        mse = F.mse_loss(reconstructed, eval_acts)
        l0 = (feature_acts != 0).float().sum(dim=-1).mean()
        
        # Explained variance
        total_var = eval_acts.var()
        residual_var = (eval_acts - reconstructed).var()
        explained_var = 1 - (residual_var / total_var)
        
        print(f"📈 SAE Quality Metrics:")
        print(f"   MSE Loss: {mse.item():.6f}")
        print(f"   L0 (avg active): {l0.item():.1f}")
        print(f"   Explained Variance: {explained_var.item():.1%}")
    
    # 2. Deception-specific metrics
    print(f"\n🔍 Deception Detection Metrics:")
    
    # Count deception-labeled features
    deception_related_labels = {
        'backdoor_behavior', 'alignment_faking',
        'factual_accuracy_deceptive', 'harmful_compliance_deceptive',
        'self_awareness_deceptive'
    }
    
    honest_related_labels = {
        'factual_accuracy_honest', 'harmful_compliance_honest',
        'self_awareness_honest'
    }
    
    deception_features = sum(
        1 for f in feature_label_dict.values()
        if f['primary_label'] in deception_related_labels
    )
    
    honest_features = sum(
        1 for f in feature_label_dict.values()
        if f['primary_label'] in honest_related_labels
    )
    
    dead_features = sum(
        1 for f in feature_label_dict.values()
        if f['primary_label'] == 'dead_feature'
    )
    
    total_features = len(feature_label_dict)
    
    print(f"   Deception-related features: {deception_features}/{total_features} ({100*deception_features/total_features:.1f}%)")
    print(f"   Honest-related features: {honest_features}/{total_features} ({100*honest_features/total_features:.1f}%)")
    print(f"   Dead features: {dead_features}/{total_features} ({100*dead_features/total_features:.1f}%)")
    
    # 3. Feature specificity - how pure are the labels?
    print(f"\n🎯 Feature Label Specificity:")
    
    specificities = []
    for feat_info in feature_label_dict.values():
        if feat_info['primary_label'] != 'dead_feature' and feat_info['label_distribution']:
            # Max probability = specificity
            max_prob = max(feat_info['label_distribution'].values())
            specificities.append(max_prob)
    
    if specificities:
        mean_specificity = np.mean(specificities)
        median_specificity = np.median(specificities)
        
        print(f"   Mean label purity: {mean_specificity:.1%}")
        print(f"   Median label purity: {median_specificity:.1%}")
        print(f"   (Higher = features are more specific to one label)")
    
    # 4. Create summary report
    summary_report = {
        'sae_quality': {
            'mse': float(mse.item()),
            'l0': float(l0.item()),
            'explained_variance': float(explained_var.item()),
        },
        'deception_detection': {
            'deception_features': deception_features,
            'honest_features': honest_features,
            'dead_features': dead_features,
            'total_features': total_features,
            'mean_specificity': float(np.mean(specificities)) if specificities else 0,
            'median_specificity': float(np.median(specificities)) if specificities else 0,
        },
        'config': SAE_CONFIG,
    }
    
    # Save report
    if 'output_dir' in locals():
        report_path = output_dir / 'deception_evaluation_report.json'
        with open(report_path, 'w') as f:
            json.dump(summary_report, f, indent=2)
        
        print(f"\n💾 Saved evaluation report to: {report_path}")
    
    print("\n✅ Evaluation complete!")
else:
    print("⚠️  Skipping evaluation - no trained SAE or labels")

## 🎨 10. Visualize Deception-Related Features

Create visualizations showing how features relate to deceptive behavior.

In [ ]:
# Visualize feature patterns
import matplotlib.pyplot as plt
import seaborn as sns

if 'feature_label_dict' in locals() and 'full_feature_acts' in locals():
    print("🎨 Creating visualizations...\n")
    
    # Create a figure with multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Deception-Focused SAE Analysis', fontsize=16, fontweight='bold')
    
    # 1. Feature activation frequency by label type
    ax = axes[0, 0]
    
    deception_activations = []
    honest_activations = []
    other_activations = []
    
    for feat_idx, feat_info in feature_label_dict.items():
        if feat_info['primary_label'] == 'dead_feature':
            continue
        
        act_count = feat_info['activation_count']
        
        if any(label in feat_info['primary_label'] for label in ['backdoor', 'alignment', 'deceptive']):
            deception_activations.append(act_count)
        elif 'honest' in feat_info['primary_label']:
            honest_activations.append(act_count)
        else:
            other_activations.append(act_count)
    
    data_to_plot = []
    labels_to_plot = []
    
    if deception_activations:
        data_to_plot.append(deception_activations)
        labels_to_plot.append('Deception')
    if honest_activations:
        data_to_plot.append(honest_activations)
        labels_to_plot.append('Honest')
    if other_activations:
        data_to_plot.append(other_activations)
        labels_to_plot.append('Other')
    
    if data_to_plot:
        ax.boxplot(data_to_plot, labels=labels_to_plot)
        ax.set_ylabel('Activation Count')
        ax.set_title('Feature Activation Frequency by Type')
        ax.set_yscale('log')
    
    # 2. Label distribution pie chart
    ax = axes[0, 1]
    
    label_counts = {}
    for feat_info in feature_label_dict.values():
        label = feat_info['primary_label']
        label_counts[label] = label_counts.get(label, 0) + 1
    
    # Only show top labels
    sorted_labels = sorted(label_counts.items(), key=lambda x: -x[1])[:8]
    labels = [l[0][:20] for l in sorted_labels]  # Truncate long labels
    sizes = [l[1] for l in sorted_labels]
    
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title('Distribution of Feature Labels')
    
    # 3. Feature specificity histogram
    ax = axes[1, 0]
    
    specificities = []
    for feat_info in feature_label_dict.values():
        if feat_info['primary_label'] != 'dead_feature' and feat_info['label_distribution']:
            max_prob = max(feat_info['label_distribution'].values())
            specificities.append(max_prob)
    
    if specificities:
        ax.hist(specificities, bins=30, edgecolor='black')
        ax.axvline(np.mean(specificities), color='red', linestyle='--',
                   label=f'Mean: {np.mean(specificities):.2f}')
        ax.set_xlabel('Label Purity (max probability)')
        ax.set_ylabel('Number of Features')
        ax.set_title('Feature Label Specificity')
        ax.legend()
    
    # 4. Top deception features activation magnitudes
    ax = axes[1, 1]
    
    # Get top deception features
    deception_feats = [
        (idx, info) for idx, info in feature_label_dict.items()
        if any(label in info['primary_label'] for label in ['backdoor', 'alignment', 'deceptive'])
        and info['primary_label'] != 'dead_feature'
    ]
    
    deception_feats.sort(key=lambda x: x[1]['activation_count'], reverse=True)
    top_n = min(15, len(deception_feats))
    
    if deception_feats:
        indices = [f"F{x[0]}" for x in deception_feats[:top_n]]
        mean_acts = [x[1]['mean_activation'] for x in deception_feats[:top_n]]
        max_acts = [x[1]['max_activation'] for x in deception_feats[:top_n]]
        
        x = np.arange(len(indices))
        width = 0.35
        
        ax.bar(x - width/2, mean_acts, width, label='Mean', alpha=0.8)
        ax.bar(x + width/2, max_acts, width, label='Max', alpha=0.8)
        
        ax.set_xlabel('Feature Index')
        ax.set_ylabel('Activation Magnitude')
        ax.set_title(f'Top {top_n} Deception Features')
        ax.set_xticks(x)
        ax.set_xticklabels(indices, rotation=45, ha='right')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    if 'output_dir' in locals():
        fig.savefig(output_dir / 'deception_analysis.png', dpi=150, bbox_inches='tight')
        print(f"💾 Saved visualization to {output_dir / 'deception_analysis.png'}")
    
else:
    print("⚠️  Skipping visualization - no feature labels available")

## 🎯 11. Summary and Next Steps

Congratulations! You've trained a deception-detecting SAE with auto-labeled features.

In [ ]:
# Print final summary
print("="*80)
print("🎉 DECEPTION-FOCUSED SAE TRAINING COMPLETE!")
print("="*80)

if 'sae' in locals() and 'feature_label_dict' in locals():
    print(f"\n✅ Successfully trained SAE on layer {SAE_CONFIG['layer']}")
    print(f"   Features: {sae_config.d_sae:,}")
    print(f"   Activations used: {SAE_CONFIG['num_activations']:,}")
    print(f"   Deception-labeled features: {deception_features}")
    print(f"   Honest-labeled features: {honest_features}")
    
    if 'output_dir' in locals():
        print(f"\n💾 Files saved to:")
        print(f"   {output_dir}")
        print(f"\n📂 Contents:")
        for file in sorted(output_dir.iterdir()):
            if file.is_file():
                size_mb = file.stat().st_size / 1e6
                print(f"   - {file.name} ({size_mb:.1f} MB)")
    
    print(f"\n🔍 Key Findings:")
    if 'summary_report' in locals():
        print(f"   - MSE: {summary_report['sae_quality']['mse']:.6f}")
        print(f"   - Explained Variance: {summary_report['sae_quality']['explained_variance']:.1%}")
        print(f"   - Mean Label Specificity: {summary_report['deception_detection']['mean_specificity']:.1%}")
    
    print(f"\n🚀 Next Steps:")
    print(f"   1. Compare with standard SAE (without deception labeling)")
    print(f"   2. Test on held-out deception examples")
    print(f"   3. Use features for steering model away from deception")
    print(f"   4. Analyze feature combinations that indicate complex deception")
    print(f"   5. Train on multiple layers to find deception circuits")
    
else:
    print("\n⚠️  Training not completed. Check the cells above for errors.")
    print("   Make sure you have:")
    print("   1. Enabled T4 GPU")
    print("   2. Downloaded deception datasets")
    print("   3. Loaded a model checkpoint")
    print("   4. Collected activations")

print("\n" + "="*80)
print("\n📚 Research Questions to Explore:")
print("   • Do auto-labeled features generalize to new deception types?")
print("   • Which layers show the strongest deception signals?")
print("   • Can we build a deception detector using feature activations?")
print("   • How do deception features evolve during model training?")
print("\n💡 Share your findings with the community!")
print("   Open an issue at: https://github.com/SolshineCode/nanochat-SAE/issues")